In [11]:
from pykafka import KafkaClient
import json
from datetime import datetime
import uuid
import time
import sys

DATA_STREAM_FILE = '../../signals/units_gps_positioning/Paris/data/data_paris_service_alpha.json'
KAFKA_HOSTS = 'localhost:9092'
KAFKA_TOPIC = 'paris_units_gps_tracking'
STREAM_SERVICE = 'alpha'

speed_up_n_times = 1

# Read coordinates from the given GeoJSON file
input_file = open(DATA_STREAM_FILE)
json_array = json.load(input_file)
coordinates = json_array['data']

# Generate an UUID
def generate_uuid():
    return uuid.uuid4()

# Initialize a Kafka producer
#client = KafkaClient(hosts=KAFKA_HOSTS)
#topic = client.topics[KAFKA_TOPIC]
#producer = topic.get_sync_producer()

# Build the messages and send them in due time to Kafka
data = {}
data['service'] = STREAM_SERVICE

def generate_checkpoint(coordinates):
    i = 0
    previous_datetime = None
    current_datetime = None

    while i < len(coordinates):
        previous_datetime = current_datetime
        current_datetime = coordinates[i]['datetime']
        data['key'] = data['service'] + '_' + str(generate_uuid())
        data['datetime'] = coordinates[i]['datetime']
        data['unit'] = coordinates[i]['unit']
        data['latitude'] = coordinates[i]['coordinates'][1]
        data['longitude'] = coordinates[i]['coordinates'][0]
        data['color'] = coordinates[i]['color']
        message = json.dumps(data)
        print(message)
        #producer.produce(message.encode('ascii'))

        if previous_datetime != None:
            sleeping_time = abs(datetime.strptime(current_datetime,"%Y-%m-%d %H:%M:%S" ) - datetime.strptime(previous_datetime,"%Y-%m-%d %H:%M:%S" )).seconds/speed_up_n_times
            time.sleep(sleeping_time)

        #if bus reaches last coordinate, start from beginning
        if i == len(coordinates)-1:
            i = 0
        else:
            i += 1

generate_checkpoint(coordinates)


{"service": "alpha", "key": "alpha_531cc098-f087-4b78-a74b-e9891412f79e", "datetime": "2020-02-13 00:00:03", "unit": 4545, "latitude": 48.8465, "longitude": 2.35156, "color": "Green"}
{"service": "alpha", "key": "alpha_6fab60ae-9074-4986-8adf-9df7f6740de2", "datetime": "2020-02-13 00:00:06", "unit": 2626, "latitude": 48.8855, "longitude": 2.28833, "color": "Green"}


KeyboardInterrupt: 

In [39]:
#!/usr/bin/python
import psycopg2
from config import config


def get_next_status_and_position(last_datetime_treated = '1900-01-01 00:00:00'):
    """ query part and vendor data from multiple tables"""
    conn = None
    query_result = []
    json_response = {}
    
    try:
        params = config()
        conn = psycopg2.connect(**params)
        cur = conn.cursor()
        cur.execute("""
            SELECT 
                json_build_object(
                    'unit', t1.unit
                    , 'lat', t1.latitude1
                    , 'lon', t1.longitude1
                    , 'status', t1.status
                    , 'color', t2.color
                ) AS data
            FROM data_id t1
            LEFT JOIN ref_status t2
                ON t1.status = t2.status
            WHERE datetime = (SELECT datetime
                FROM data_id
                WHERE datetime > '"""+last_datetime_treated+"""'
                ORDER BY datetime
                LIMIT 1);

        """)
        row = cur.fetchone()

        while row is not None:
            # message = row[0]
            query_result.append(row[0])
            row = cur.fetchone()
        
        data = {}
        data['datetime'] = last_datetime_treated
        data['service'] = 'alpha'
        data['data'] = query_result
        json_response = json.dumps(data)

        cur.close()
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)
    finally:
        if conn is not None:
            conn.close() 
            
    return json_response
            
get_next_status_and_position('2018-01-01 00:00:00')

'{"datetime": "2018-01-01 00:00:00", "service": "alpha", "data": [{"unit": 630, "lat": 48.7455213, "lon": 2.45165, "status": 1, "color": "Red"}, {"unit": 99, "lat": 48.7966356, "lon": 2.5021718, "status": 1, "color": "Red"}, {"unit": 2419, "lat": 48.8466399, "lon": 2.3510102, "status": 1, "color": "Red"}, {"unit": 4010, "lat": 48.8312252, "lon": 2.3179593, "status": 1, "color": "Red"}, {"unit": 478, "lat": 48.8995701, "lon": 2.2020856, "status": 1, "color": "Red"}, {"unit": 2695, "lat": 48.8129761, "lon": 2.3281774, "status": 1, "color": "Red"}, {"unit": 2576, "lat": 48.8721544, "lon": 2.4037482, "status": 1, "color": "Red"}, {"unit": 1485, "lat": 48.8615397, "lon": 2.3057038, "status": 1, "color": "Red"}, {"unit": 2733, "lat": 48.8812401, "lon": 2.3682868, "status": 1, "color": "Red"}, {"unit": 2837, "lat": 48.9201291, "lon": 2.440259, "status": 1, "color": "Red"}, {"unit": 4226, "lat": 48.9201291, "lon": 2.440259, "status": 1, "color": "Red"}, {"unit": 2376, "lat": 48.7865059, "lon":

In [34]:
import time
from datetime import datetime
from pykafka import KafkaClient
import json

previous_datetime = None
current_datetime = '1900-01-01T00:00:00'
speed_up_n_times = 1

KAFKA_HOSTS = 'localhost:9092'
KAFKA_TOPIC = 'paris_units_gps_tracking'
STREAM_SERVICE = 'alpha'

# Initialize a Kafka producer
client = KafkaClient(hosts=KAFKA_HOSTS)
topic = client.topics[KAFKA_TOPIC]
producer = topic.get_sync_producer()
    
while 1:
    previous_datetime = current_datetime

    data = get_next_status_and_position(current_datetime)
    current_datetime = data[0]['datetime']
        
    if previous_datetime != None:
        sleeping_time = abs(datetime.strptime(current_datetime,"%Y-%m-%dT%H:%M:%S" ) - datetime.strptime(previous_datetime,"%Y-%m-%dT%H:%M:%S" )).seconds/speed_up_n_times
        time.sleep(sleeping_time)
        
    for state in data:
        producer.produce(json.dumps(state).encode('ascii'))

{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:00:00', 'unit': 630, 'latitude': 48.7455213, 'longitude': 2.45165, 'status': 1, 'color': 'Red'}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:00:00', 'unit': 99, 'latitude': 48.7966356, 'longitude': 2.5021718, 'status': 1, 'color': 'Red'}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:00:00', 'unit': 2419, 'latitude': 48.8466399, 'longitude': 2.3510102, 'status': 1, 'color': 'Red'}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:00:00', 'unit': 4010, 'latitude': 48.8312252, 'longitude': 2.3179593, 'status': 1, 'color': 'Red'}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:00:00', 'unit': 478, 'latitude': 48.8995701, 'longitude': 2.2020856, 'status': 1, 'color': 'Red'}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:00:00', 'unit': 2695, 'latitude': 48.8129761, 'longitude': 2.3281774, 'status': 1, 'color': 'Red'}
{'service': 'alp

{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:00:08', 'unit': 3859, 'latitude': None, 'longitude': None, 'status': 9, 'color': 'Green'}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:00:24', 'unit': 4005, 'latitude': None, 'longitude': None, 'status': 9, 'color': 'Green'}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:02:20', 'unit': 3910, 'latitude': None, 'longitude': None, 'status': 116, 'color': None}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:02:22', 'unit': 3834, 'latitude': None, 'longitude': None, 'status': 103, 'color': 'Orange'}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:02:24', 'unit': 3834, 'latitude': 48.8373818, 'longitude': 2.3437573, 'status': 102, 'color': 'Orange'}
{'service': 'alpha', 'key': 'the_key', 'datetime': '2019-01-01T00:02:35', 'unit': 3910, 'latitude': None, 'longitude': None, 'status': 9, 'color': 'Green'}
{'service': 'alpha', 'key': 'the_key', 'datetime

KeyboardInterrupt: 

In [35]:
import argparse